# Исследование поездок такси Нью-Йорка (NYC Taxi Analysis)

### Описание проекта
Этот ноутбук предназначен для анализа данных о поездках желтого такси в Нью-Йорке. Исследуем спрос, выручку и операционную эффективность в зависимости от времени, расстояния и географии

**Используемый датасет:** `samples.nyctaxi.trips` (содержит данные о времени посадки/высадки, дистанции, стоимости и локациях поездок в Нью-Йорке)

### Инструкция для пользования:
1. Выберите интересующий **диапазон дат** в виджетах сверху.
2. Укажите **уровень агрегации** (день, неделя или месяц) для графиков динамики.
3. Отфильтруйте данные по **району (Zip-код)** и **максимальной дистанции**.
4. Нажмите **"Run All"**, чтобы обновить все аналитические выводы.

In [0]:
display(spark.read.table("samples.nyctaxi.trips").limit(5))

In [0]:
# Получаем ВСЕ уникальные Zip-коды и сортируем их
all_zips_df = spark.read.table("samples.nyctaxi.trips") \
    .select("pickup_zip") \
    .distinct() \
    .sort("pickup_zip")

# Собираем в список для использования при передаче в UI
zip_list = [str(row[0]) for row in all_zips_df.collect() if row[0] is not None]
zip_list.insert(0, "All")

In [0]:
dbutils.widgets.text("start_date", "2016-01-01", "1. Дата начала")
dbutils.widgets.text("end_date", "2016-01-31", "2. Дата конца")
dbutils.widgets.dropdown("pickup_zip", "All", zip_list, "3. Район (Zip)")
dbutils.widgets.dropdown("aggregation_level", "Day", ["Day", "Week", "Month"], "4. Агрегация по времени")
dbutils.widgets.text("max_distance", "30", "5. Макс. дистанция (мили)")


In [0]:
from pyspark.sql.functions import col, unix_timestamp, round, when, lit

# Извлекаем текущие значения из виджетов
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
selected_zip = dbutils.widgets.get("pickup_zip")
max_dist = float(dbutils.widgets.get("max_distance"))

# Загружаем таблицу и применяем фильтры
df_raw = spark.read.table("samples.nyctaxi.trips")

# Фильтруем по датам и дистанции
df_filtered = df_raw.filter(
    (col("tpep_pickup_datetime").cast("date") >= start_date) & 
    (col("tpep_pickup_datetime").cast("date") <= end_date) &
    (col("trip_distance") <= max_dist)
)

# Если в виджете выбран конкретный район, фильтруем и по нему
if selected_zip != "All":
    df_filtered = df_filtered.filter(col("pickup_zip") == selected_zip)

# Очистка от "грязных" данных
# Убираем нулевые дистанции, нулевые суммы и некорректное время
df_cleaned = df_filtered.filter(
    (col("trip_distance") > 0) & 
    (col("fare_amount") > 0) & 
    (col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
)

display(df_cleaned.limit(10))

In [0]:
from pyspark.sql.functions import hour, dayofweek, date_format, to_date, unix_timestamp, round, col, when

# Добавляем новые колонки для последующих визуалов в наш очищенный датафрейм
df_features = df_cleaned.withColumn(
    # Длительность поездки в минутах
    "trip_duration_minutes", 
    round((unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60,2)
).withColumn(
    # Чистая дата (без времени)
    "pickup_date", to_date("tpep_pickup_datetime")
).withColumn(
    # Час посадки (0-23)
    "pickup_hour", hour("tpep_pickup_datetime")
).withColumn(
    # День недели
    "weekday", date_format("tpep_pickup_datetime", "EEEE")
).withColumn(
    # Выручка на одну милю 
    "revenue_per_mile", round(col("fare_amount") / col("trip_distance"), 2)
).withColumn(
    # Категоризация поездок по дистанции
    "trip_category", 
    when(col("trip_distance") < 2, "Short (< 2 mi)")
    .when((col("trip_distance") >= 2) & (col("trip_distance") <= 10), "Medium (2-10 mi)")
    .otherwise("Long (> 10 mi)")
)

# Проверяем результат на небольшом наборе колонок
display(df_features.select(
    "tpep_pickup_datetime", "trip_distance", "fare_amount", 
    "trip_duration_minutes", "revenue_per_mile", "trip_category"
).limit(10))

In [0]:
import pyspark.sql.functions as F

# Получаем уровень агрегации по времени
agg_level = dbutils.widgets.get("aggregation_level")

# Группируем и агрегируем с округлением, выведем два графики - график поездок и график выручки
demand_df = df_features.groupBy(
    F.date_trunc(agg_level, "pickup_date").alias("period")
).agg(
    F.count("*").alias("trips_count"),
    F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
    F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance")
).orderBy("period")

display(demand_df)

Databricks visualization. Run in Databricks to view.

In [0]:

# Группируем по часу посадки и считаем количество поездок
peaks_df = df_features.groupBy("pickup_hour").agg(
    F.count("*").alias("trips_count"),
    F.round(F.sum("fare_amount"), 2).alias("hourly_revenue")
).orderBy("pickup_hour")

display(peaks_df)

Databricks visualization. Run in Databricks to view.

In [0]:
# Смотри выручку в зависимости от расстояния поездки
# Берем выборку 20% данных для визуализации (чтобы график был нормально воспринимаемым)
dist_rev_df = df_features.select(
    F.round("trip_distance", 2).alias("distance"), 
    F.round("fare_amount", 2).alias("revenue")
).sample(withReplacement=False, fraction=0.2, seed=1)

display(dist_rev_df)

Databricks visualization. Run in Databricks to view.

In [0]:
# Группируем по часу и считаем среднюю выручку на милю (округляем до 2 знаков)
efficiency_df = df_features.groupBy("pickup_hour").agg(
    F.round(F.avg("revenue_per_mile"), 2).alias("avg_revenue_per_mile"),
    F.count("*").alias("trip_count")
).orderBy("pickup_hour")

display(efficiency_df)

Databricks visualization. Run in Databricks to view.

In [0]:
# Группируем по категории поездки
market_df = df_features.groupBy("trip_category").agg(
    F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
    F.count("*").alias("trip_count")
).orderBy(F.desc("total_revenue"))

display(market_df)

Databricks visualization. Run in Databricks to view.

## Итоговые выводы
1. **Динамика:** В выбранном периоде (январь 2016 года) наблюдается стабильный спрос на уровне около 300-400 поздок в день с выручкой около 4000 долларов. Исключение - 23-24.01.2026, когда количество поездок упало в 3-4 раза, что было вызвано снежной бурей в данные дни.
2. **Пики:** Самое загруженное время — это 18-20 часов вечера, что соответствует вечернему часу пик. Минимальная загрузка - 3-5 часов утра.
3. **Эффективность:** Самый высокий доход на милю (RPM) зафиксирован в 7 часов утра, что делает это время наиболее выгодным для водителей (доход на милю в 1.5-2 раза выше, чем в другие часы дня).
4. **Сегментация:** Основную долю выручки приносят поездки на короткое расстояние (до 2 миль) - почти половина всей выручки.

